# 03 - Do modelo à ação: risco de churn + mensagem de retenção com Claude

Modelo escolhido na etapa 02: **Regressão Logística** (maior recall, AUC equivalente à Random Forest e coeficientes fáceis de explicar para o negócio).

Aqui o modelo substitui as regras fixas do pipeline antigo (`UsageScore < 20` etc.): cada cliente recebe uma **probabilidade de cancelar**, vira uma faixa de risco, e a API do Claude escreve a mensagem de retenção com base nessa faixa e nos fatores do cliente.

In [1]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('../data/telco_tratada.csv')
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']
num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat = [c for c in X.columns if c not in num]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

modelo = Pipeline([
    ('prep', ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000)),
]).fit(X_tr, y_tr)

Path('../models').mkdir(exist_ok=True)
joblib.dump(modelo, '../models/modelo_churn.joblib')
print('Modelo salvo em models/modelo_churn.joblib')

Modelo salvo em models/modelo_churn.joblib


## 1. Probabilidade e faixa de risco (clientes do teste)

Faixas usadas: **Alto** (≥ 70%), **Médio** (40% a 70%) e **Baixo** (< 40%).
Como o modelo foi treinado com `class_weight='balanced'`, as probabilidades ficam "infladas" para o lado do churn (é o preço de ter recall alto). Por isso as faixas são checadas contra a **taxa real de cancelamento** de cada uma, logo abaixo.

In [2]:
teste = df.loc[X_te.index].copy()
teste['prob_churn'] = modelo.predict_proba(X_te)[:, 1]
teste['risco'] = pd.cut(teste['prob_churn'], [0, .4, .7, 1], labels=['Baixo', 'Médio', 'Alto'], include_lowest=True)

resumo = teste.groupby('risco', observed=True).agg(clientes=('customerID', 'size'), churn_real=('Churn', 'mean'))
resumo['churn_real'] = (resumo['churn_real'] * 100).round(1).astype(str) + '%'
resumo

,clientes,churn_real
risco,,
Baixo,686,7.0%
Médio,343,27.7%
Alto,378,61.1%


## 2. Mensagem de retenção com a API do Claude

A chave fica só na variável de ambiente `ANTHROPIC_API_KEY` (nunca no código). Sem a chave, o notebook roda normalmente e apenas pula esta parte.
Para não gastar à toa, geramos mensagens só para uma amostra: os 3 clientes de maior risco e 1 cliente de cada uma das outras faixas.

In [3]:
def gerar_mensagem(cliente: dict, client) -> str:
    prompt = f"""Você é especialista em Customer Success de uma operadora de internet e telefonia.
Escreva uma mensagem de retenção curta, em português, para este cliente:

- Risco de cancelamento previsto pelo modelo: {cliente['risco']} ({cliente['prob_churn']:.0%})
- Meses como cliente: {cliente['tenure']}
- Contrato: {cliente['Contract']}
- Internet: {cliente['InternetService']} | Suporte técnico: {cliente['TechSupport']}
- Forma de pagamento: {cliente['PaymentMethod']}
- Mensalidade: US$ {cliente['MonthlyCharges']:.2f}

Regras:
- Risco Alto: ofereça um benefício concreto ligado ao perfil (ex.: migrar para contrato anual com desconto, suporte técnico incluso, débito automático).
- Risco Médio: incentive o uso e mostre um benefício que o cliente ainda não usa.
- Risco Baixo: agradeça e reconheça o tempo de casa.
- Máximo de 3 frases, tom humano e direto, sem prometer valores que não estão nos dados."""
    resp = client.messages.create(model='claude-sonnet-5', max_tokens=300,
                                  messages=[{'role': 'user', 'content': prompt}])
    return resp.content[0].text.strip()


amostra = pd.concat([
    teste.sort_values('prob_churn', ascending=False).head(3),
    teste[teste['risco'] == 'Médio'].head(1),
    teste[teste['risco'] == 'Baixo'].head(1),
])

if os.environ.get('ANTHROPIC_API_KEY'):
    import anthropic
    client = anthropic.Anthropic()
    amostra['mensagem'] = [gerar_mensagem(c, client) for c in amostra.to_dict(orient='records')]
else:
    amostra['mensagem'] = '(defina ANTHROPIC_API_KEY para gerar)'
    print('ANTHROPIC_API_KEY não definida: mensagens não geradas.')

amostra[['customerID', 'prob_churn', 'risco', 'Contract', 'tenure', 'mensagem']]

ANTHROPIC_API_KEY não definida: mensagens não geradas.


,customerID,prob_churn,risco,Contract,tenure,mensagem
3154,5150-ITWWB,0.934325,Alto,Month-to-month,3,(defina ANTHROPIC_API_KEY para gerar)
3374,5178-LMXOP,0.933925,Alto,Month-to-month,1,(defina ANTHROPIC_API_KEY para gerar)
2626,6861-XWTWQ,0.924503,Alto,Month-to-month,7,(defina ANTHROPIC_API_KEY para gerar)
3715,2839-RFSQE,0.401228,Médio,Month-to-month,2,(defina ANTHROPIC_API_KEY para gerar)
971,0604-THJFP,0.050761,Baixo,Two year,59,(defina ANTHROPIC_API_KEY para gerar)


## 3. Load: salvar o resultado

In [4]:
amostra.to_csv('../data/mensagens_retencao.csv', index=False)
teste[['customerID', 'prob_churn', 'risco', 'Churn']].to_csv('../data/risco_clientes_teste.csv', index=False)
print('Arquivos salvos em data/')

Arquivos salvos em data/
